In [2]:
from supabase import create_client, Client
from dotenv import load_dotenv
import os


load_dotenv()

url= os.getenv("SUPABASE_URL")
key= os.getenv("SUPABASE_KEY")

supabase: Client = create_client(url,key)

try:
    response = supabase.table("movies2").select("*", count="exact").execute()
    print(f"Total movies: {response.count}")
    print(response.data)
except Exception as e:
    print(e)



Total movies: 9083
[{'Id': '0', 'Title': 'Napoleon', 'Summary': 'An epic that details the checkered rise and fall of French Emperor Napoleon Bonaparte and his relentless journey to power through the prism of his addictive, volatile relationship with his wife, Josephine.', 'Director': 'Ridley Scott', 'Writer': 'David Scarpa', 'Main Genres': 'Action,Adventure,Biography', 'Motion Picture Rating': 'R', 'Release Year': 2023, 'Runtime (Minutes)': '158.0', 'Rating (Out of 10)': 6.7, 'Number of Ratings (in thousands)': 38, 'Budget (in millions)': None, 'Gross in US & Canada (in millions)': '37.514', 'Gross worldwide (in millions)': '84.968', 'Opening Weekend in US & Canada': '11.26.2023', 'Gross Opening Weekend (in millions)': '20.639'}, {'Id': '1', 'Title': 'The Hunger Games: The Ballad of Songbirds & Snakes', 'Summary': 'Coriolanus Snow mentors and develops feelings for the female District 12 tribute during the 10th Hunger Games.', 'Director': 'Francis Lawrence', 'Writer': 'Michael Lesslie,M

In [5]:
import os
import psycopg2
import pandas as pd
from psycopg2.extras import RealDictCursor
from supabase import create_client, Client
from dotenv import load_dotenv
from contextlib import contextmanager
from typing import Optional, Dict, Any, Union
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

load_dotenv()

class DatabaseManager:
    """Hybrid database manager for Supabase + psycopg2"""
    
    def __init__(self):
        # Supabase client for web features
        self.supabase_url = os.getenv("SUPABASE_URL")
        self.supabase_key = os.getenv("SUPABASE_KEY")
        self.supabase: Optional[Client] = None
        
        # PostgreSQL connection params
        self.pg_params = {
            "user": os.getenv("user"),
            "password": os.getenv("password"),
            "host": os.getenv("host"),
            "port": os.getenv("port"),
            "dbname": os.getenv("dbname")
        }
        
    def init_supabase(self) -> Client:
        """Initialize Supabase client (for web features)"""
        if not self.supabase:
            self.supabase = create_client(self.supabase_url, self.supabase_key)
            logger.info("✅ Supabase client initialized")
        return self.supabase
    
    @contextmanager
    def get_connection(self, cursor_factory=RealDictCursor):
        """Context manager for psycopg2 connections (for complex queries)"""
        conn = None
        try:
            conn = psycopg2.connect(**self.pg_params, cursor_factory=cursor_factory)
            logger.debug("Database connection established")
            yield conn
        except Exception as e:
            logger.error(f"Database connection error: {e}")
            raise
        finally:
            if conn:
                conn.close()
                logger.debug("Database connection closed")
    
    @contextmanager
    def get_cursor(self, cursor_factory=RealDictCursor):
        """Get a cursor for executing queries"""
        with self.get_connection(cursor_factory) as conn:
            cur = conn.cursor()
            try:
                yield cur
                conn.commit()
            except Exception:
                conn.rollback()
                raise
            finally:
                cur.close()
    
    # ========== DATA SCIENCE METHODS ==========
    
    def query_to_dataframe(self, sql: str, params: tuple = None) -> pd.DataFrame:
        """Execute SQL and return pandas DataFrame (for data science)"""
        with self.get_connection(cursor_factory=None) as conn:
            return pd.read_sql(sql, conn, params=params)
    
    def get_movie_stats(self) -> pd.DataFrame:
        """Get movie statistics for analysis"""
        sql = """
        WITH movie_stats AS (
            SELECT 
                genre,
                COUNT(*) as movie_count,
                AVG(rating) as avg_rating,
                PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY rating) as median_rating,
                MIN(release_year) as oldest_movie,
                MAX(release_year) as newest_movie
            FROM movies2
            GROUP BY genre
        )
        SELECT * FROM movie_stats
        ORDER BY movie_count DESC
        """
        return self.query_to_dataframe(sql)
    
    
    
    # ========== WEB APP METHODS ==========
    
    def get_public_movies(self, limit: int = 50):
        """Get movies for public view (uses Supabase RLS)"""
        supabase = self.init_supabase()
        return supabase.table("movies2")\
            .select("*")\
            .limit(limit)\
            .execute()
    
    def get_user_movies(self, user_id: str):
        """Get movies for specific user (respects RLS)"""
        supabase = self.init_supabase()
        return supabase.table("movies2")\
            .select("*")\
            .eq("user_id", user_id)\
            .execute()
    
    def add_movie(self, movie_data: Dict[str, Any], user_id: str = None):
        """Add a new movie"""
        supabase = self.init_supabase()
        if user_id:
            movie_data["user_id"] = user_id
        return supabase.table("movies2").insert(movie_data).execute()
    

# Create singleton instance
db = DatabaseManager()